# 00_environment_and_data — shared external storage

목적: 후속 모델 생성 단계가 공유할 조건을 설정·저장하고 실행 환경과 계획을 확인합니다.

UCFace explicit settings take priority; otherwise FaceX-Zoo conventional training.
Target-only FP32 zero-radicand safeguard; 112×112 inputs; full training lists and final batches.
VGGFace2: provided five-point landmarks + original FaceX-Zoo cropper.
TinyFace/SurvFace/SCface: user-approved resize-only, default bicubic; no face re-detection.
Existing failed-condition checkpoints are preserved as records. 00 saves settings; 01–04 start with EXECUTE=False.


## 1. 사용자 설정
기본 5 seed와 네 head입니다. 데이터/완료 checkpoint를 수정하지 않습니다.

선택지·범위가 적힌 변수만 필요한 조건으로 조정합니다. 공통 변경은 00과 같은 SETTINGS_PATH를 사용하는 후속 단계에 일관되게 반영합니다.


In [6]:
# 목적: 공유 경로와 backbone·head·seed·학습 조건을 정해 후속 노트북의 공통 설정을 구성합니다.

from pathlib import Path
import sys, json, os
BASE_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "fine-tune/modules").is_dir())  # 저장소 위치 자동 탐색값입니다. 직접 변경하지 않습니다.
DATASET_ROOT = Path(os.environ.get("FACE_DATASET_ROOT", "F:/face_datasets" if os.name == "nt" else "/data/face_datasets"))  # 공유 데이터 루트; 환경변수 FACE_DATASET_ROOT 또는 이 Path를 실제 경로로 지정합니다.
MODEL_ROOT = Path(os.environ.get("FACE_MODEL_ROOT", "F:/models" if os.name == "nt" else "/data/models"))  # 공유 모델 루트; 환경변수 FACE_MODEL_ROOT 또는 이 Path를 실제 경로로 지정합니다.
SETTINGS_PATH = DATASET_ROOT / ".state/facex/shared_v1_boundary/notebook_settings_mobilefacenet.json"  # 00에서 저장하고 01~04에서 함께 읽을 JSON입니다. backbone/실험별로 파일을 구분합니다.
for path in (BASE_DIR, BASE_DIR / "fine-tune"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
from modules.notebook_workflow import resolve_settings, save_settings, plan_matrix
# SEEDS = [8972, 8973, 8974, 8975, 8976]  # 중복 없는 정수 목록(0 <= seed < 2**32); 일부 seed만 선택 가능하며 후속 단계도 동일하게 맞춥니다.
SEEDS = [8972]  # 중복 없는 정수 목록(0 <= seed < 2**32); 일부 seed만 선택 가능하며 후속 단계도 동일하게 맞춥니다.
HEADS = ["arcface", "adaface", "magface", "cosface"]  # "arcface", "adaface", "magface", "cosface" 중 중복 없이 선택; 02 부모는 ArcFace로 고정입니다.
PARENT_SEED = 8972  # 공통 부모 seed(0 <= seed < 2**32 정수); 후속 head/seed가 공유하므로 02~04에서 유지합니다.
PRETRAIN_EPOCHS = 40  # 부모 학습 epoch 수(양의 정수, 기본 40); 재사용 부모의 완료 epoch와 일치해야 합니다.
PRETRAIN_LR = 0.001  # parent의 기존 LR을 별도 유지합니다. downstream 논문 LR과 구분합니다.
PARENT_CHECKPOINT = None  # None=02의 고정 경로; 다른 완료 부모를 쓰려면 정확한 경로와 PARENT_SHA256를 함께 지정합니다.
PARENT_SHA256 = None  # None 또는 지정한 부모 파일의 64자리 SHA-256; 경로와 함께 지정하며 학습 조건도 검증합니다.
FINETUNE_DATASETS = ["survface", "tinyface", "scface"]  # "survface", "tinyface", "scface" 중 선택합니다. LFW/RFW는 평가 전용입니다.
# 원본 데이터·공유 전처리·부모·과거 실패 기록은 이 보존 옵션의 삭제 대상이 아닙니다.
KEEP_RAW_RESULTS = True  # True=이번 학습의 주기별 checkpoint 보존, False=학습 중 오래된 중간 checkpoint 정리.
DATA_INTEGRITY_MODE = "content_verified_reuse"  # "content_verified_reuse"=현재 전체 hash 일치 시 decode 검사 재사용, "full"=매번 hash+decode.
# 검사 재사용에도 매번 원본/전처리 이미지의 바이트 hash를 확인합니다.
DEVICE = "cuda:0"  # "cuda:0" 또는 사용 가능한 GPU 번호("cuda:1" 등); 정식 학습은 CUDA 필수이며 CPU fallback은 없습니다.
NUM_WORKERS = 4  # DataLoader worker 수(0~12 정수); 0=현재 프로세스에서 로딩, 큰 값은 CPU/RAM을 더 사용합니다.
PREFETCH_FACTOR = 4  # worker당 미리 읽을 batch 수(1 이상 정수); NUM_WORKERS>0일 때 적용합니다.
PERSISTENT_WORKERS = True  # True=epoch 사이 worker 유지, False=다시 생성; NUM_WORKERS=0이면 적용되지 않습니다.
TORCH_THREADS = 2  # PyTorch CPU 연산 thread 수(1 이상 정수); GPU 수나 DataLoader worker 수와 다릅니다.
LOSS_SYNC_BATCHES = 20  # loss 집계값을 CPU로 가져오는 batch 간격(1 이상 정수); 매 batch 학습 업데이트는 유지합니다.
RECOVERY_MINUTES = 10  # 재개용 checkpoint 저장 간격(분, 0보다 큰 값); 짧을수록 복구 간격과 디스크 쓰기가 촘촘해집니다.
REUSE_COMPLETED = True  # True=동일 조건의 검증된 완료 결과 재사용; False=완료 경로가 있으면 중단하며 덮어쓰지 않습니다.
# 아래 현재 변수값이 저장된 설정/YAML보다 우선합니다.
from modules.shared_storage import external_settings
RUN_NAMESPACE = None  # None=backbone별 기본 이름; 조건 변경 시 새 이름(영문·숫자·"_"·"-")을 지정합니다.
defaults = external_settings(BASE_DIR, DATASET_ROOT, MODEL_ROOT, namespace=RUN_NAMESPACE)
DATASET = dict(defaults["dataset"])  # 임시 기본값입니다. 실제 DATASET은 아래 backbone 반영 후 다시 생성합니다.
BACKBONE = dict(defaults["backbone"])  # 00에서 MobileFaceNet/Resnet IR-50 선택; 1024D·dropout 0.6·out_h/out_w=7 조건을 유지합니다.
# Resnet IR-50 선택 예: 아래 차원 고정 전에 BACKBONE.update(type="Resnet", num_layers=50, mode="ir")를 추가합니다.
# backbone별 SETTINGS_PATH와 부모 checkpoint를 구분하고 후속 노트북에서도 같은 파일을 선택합니다.
BACKBONE.update(feat_dim=1024)  # 두 backbone 모두 1024D로 고정합니다.
# 정식 fine-tune: optimizer=adam, lr=MobileFaceNet 0.001 / ResNet-50·100 0.0001, epoches=40, milestones=[10,20,30], lr_decay_ratio=0.1 고정.
# baseline batch_size=64는 MobileFaceNet·ResNet-50/100 공통; save_freq는 1 이상 epoch, drop_last=False 유지.
TRAINING = dict(defaults["training"])  # 학습/저장 설정 사전입니다. 위 고정 학습 조건과 조절 가능한 저장 간격을 구분합니다.
TRAINING.update(save_freq=1, drop_last=False)  # 저장 간격(epoch)은 조절 가능, 마지막 batch를 사용하는 False는 유지.
from modules.backbone_factory import canonical_backbone_type, baseline_learning_rate
BACKBONE["type"] = canonical_backbone_type(BACKBONE["type"])  # 대소문자를 구분하지 않고 판별합니다.
TRAINING["lr"] = baseline_learning_rate(BACKBONE)
TRAINING["batch_size"] = 64  # UCFace softmax baseline 공통 batch. 과거 batch 16 checkpoint와 혼용하지 않습니다.
# backbone과 RUN_NAMESPACE로 실제 DATASET/PATHS를 다시 만듭니다. 경로 조정은 이 아래에서 합니다.
defaults = external_settings(BASE_DIR, DATASET_ROOT, MODEL_ROOT, backbone=BACKBONE, namespace=RUN_NAMESPACE)
DATASET = dict(defaults["dataset"])  # VGGFace2 경로·입력 설정입니다. 112×112와 pixel_mean=127.5/pixel_std=128.0 조건을 유지합니다.
PATHS = dict(defaults["paths"])  # checkpoint/export/log/report/cache 경로; 00에서 namespace별로 정하고 후속 단계가 이어받습니다.
FINETUNE_PROFILES = defaults["finetune_profiles"]  # 외부 protocol의 분할·경로·hash입니다. 분할 인물/수량/hash를 여기서 임의로 고치지 않습니다.
HEAD_PARAMETERS = defaults["heads"]  # head별 margin/scale 기준값입니다. 정식 fine-tune은 이를 검증하므로 head 선택은 HEADS에서 합니다.
PARENT_HEAD = dict(type="ArcFace", margin_arc=0.35, margin_am=0.0, scale=32.0)  # 부모는 ArcFace, margin_arc=0.35/margin_am=0/scale=32로 고정합니다.
# augmentation 종류와 연구자가 정한 수치를 구분합니다.
# flip_probability/affine_probability: 0~1, rotation_degrees: 0 이상, scale_range=[low,high]: 0<low<=high.
# border_mode="reflect"/"constant", center="image_center" 고정; 현재 수치를 UCFace 명시값으로 간주하지 않습니다.
AUGMENTATION = dict(flip_probability=0.5, affine_probability=0.5,  # 증강 수치 변경은 별도 학습 조건입니다. 새 namespace와 부모 학습으로 구분합니다.
    rotation_degrees=10.0, scale_range=[0.95, 1.05], border_mode="reflect", center="image_center")
# resize_only는 승인된 연구 조건입니다. 보간법 변경은 새 namespace에서 준비합니다.
ALIGNMENT = defaults["alignment"]  # VGGFace2=official_landmarks, 저해상도=resize_only; 보간법은 bicubic/bilinear 중 선택합니다.
# 위 변수들을 실제 모듈 설정으로 묶어 검증합니다. 아래 전달부는 직접 편집할 필요가 없습니다.
settings = resolve_settings(BASE_DIR, SETTINGS_PATH, overrides={
 "storage":defaults["storage"],
 "data_integrity":dict(mode=DATA_INTEGRITY_MODE),
 "project":dict(seed=SEEDS[0],seed_list=SEEDS,keep_raw_results=KEEP_RAW_RESULTS,device=DEVICE),
 "campaign":dict(heads=HEADS,reuse_completed=REUSE_COMPLETED,parent_seed=PARENT_SEED,
    pretrain_epochs=PRETRAIN_EPOCHS,pretrain_lr=PRETRAIN_LR,parent_checkpoint=PARENT_CHECKPOINT,parent_sha256=PARENT_SHA256,
    finetune_datasets=FINETUNE_DATASETS,handoff_path=defaults["campaign"]["handoff_path"]),
 "finetune_profiles":FINETUNE_PROFILES,"dataset":DATASET,"backbone":BACKBONE,"heads":HEAD_PARAMETERS,
 "parent_head":PARENT_HEAD,"augmentation":AUGMENTATION,"alignment":ALIGNMENT,
 "training":{**TRAINING,"num_workers":NUM_WORKERS,"prefetch_factor":PREFETCH_FACTOR,
    "persistent_workers":PERSISTENT_WORKERS,"torch_threads":TORCH_THREADS,
    "recovery_minutes":RECOVERY_MINUTES,"loss_sync_batches":LOSS_SYNC_BATCHES},"paths":PATHS})


## 2. 설정 저장

In [7]:
# 목적: 공통 설정을 SETTINGS_PATH에 저장하여 01~04가 같은 조건을 읽도록 합니다.

print(save_settings(SETTINGS_PATH, settings))

F:\face_datasets\.state\facex\shared_v1_boundary\notebook_settings_mobilefacenet.json


## 3. 실제 환경 확인

In [8]:
# 목적: Python·CUDA·장치와 데이터 준비 상태를 조회해 실행 환경을 확인합니다.

from modules.notebook_workflow import inspect_environment
environment = inspect_environment(BASE_DIR, settings)
print(json.dumps(environment, ensure_ascii=False, indent=2))

{
  "python": "3.11.9",
  "torch": "2.7.1+cu118",
  "cuda_available": true,
  "gpu": "NVIDIA GeForce GTX 1080 Ti",
  "branch": "refacto1",
  "branch_matches": false,
  "train_archive": {
    "exists": true,
    "bytes": 37909212476
  },
  "test_archive": {
    "exists": true,
    "bytes": 2028325313
  },
  "meta_csv": {
    "exists": true,
    "bytes": 335000
  },
  "free_bytes": 1198568292352,
  "training_identities": 8631,
  "identity_count_matches": true,
  "archive_integrity": "not_checked",
  "all_images_decode": "not_checked",
  "readiness": "partial_environment_check_only"
}


## 4. 실행 계획

In [9]:
# 목적: 선택한 조건으로 만들어질 부모 학습·fine-tune·export의 run 목록과 개수를 확인합니다.

for stage in ("pretrain","finetune","export"):
    plan=plan_matrix(settings,stage)
    print(stage,len(plan),[r["run_id"] for r in plan])
print("공통 부모 1개. 선택 데이터셋별 네 head × SEEDS. 공유 부모는 본 연구 선택입니다.")

pretrain 1 ['pretrain_arcface_seed8972']
finetune 12 ['finetune_survface_arcface_seed8972', 'finetune_survface_adaface_seed8972', 'finetune_survface_magface_seed8972', 'finetune_survface_cosface_seed8972', 'finetune_tinyface_arcface_seed8972', 'finetune_tinyface_adaface_seed8972', 'finetune_tinyface_magface_seed8972', 'finetune_tinyface_cosface_seed8972', 'finetune_scface_arcface_seed8972', 'finetune_scface_adaface_seed8972', 'finetune_scface_magface_seed8972', 'finetune_scface_cosface_seed8972']
export 12 ['export_survface_arcface_seed8972', 'export_survface_adaface_seed8972', 'export_survface_magface_seed8972', 'export_survface_cosface_seed8972', 'export_tinyface_arcface_seed8972', 'export_tinyface_adaface_seed8972', 'export_tinyface_magface_seed8972', 'export_tinyface_cosface_seed8972', 'export_scface_arcface_seed8972', 'export_scface_adaface_seed8972', 'export_scface_magface_seed8972', 'export_scface_cosface_seed8972']
공통 부모 1개. 선택 데이터셋별 네 head × SEEDS. 공유 부모는 본 연구 선택입니다.


## 5. 확인 범위

In [10]:
# 목적: 환경 점검으로 확인한 범위와 미실행 작업을 구분하고 실행 안내 위치를 표시합니다.

print("부분 환경 점검입니다. 압축 전체 무결성·전체 이미지 decode·학습·FIQA 평가는 미실행입니다.")
print("안내:", BASE_DIR / "fine-tune/README.md")

부분 환경 점검입니다. 압축 전체 무결성·전체 이미지 decode·학습·FIQA 평가는 미실행입니다.
안내: f:\FaceX-Zoo\fine-tune\README.md


새 연구 run은 target-only FP32 zero-radicand safeguard를 처음부터 사용합니다. 기존 parent/checkpoint는 새 정책으로 이어 학습하지 않습니다. 전처리 검증은 최초 hash+decode 후 현재 바이트 hash가 모두 일치할 때만 decode 결과를 재사용합니다. 검증 동안과 학습 중에는 데이터를 변경하지 않습니다.
분할은 기존 고정 후보를 먼저 평가하며, fit/safety 표본 수 분석은 ronbun에서 동일 모델·codec을 고정한 별도 단계로 수행합니다. FR/calibration 비율 비교는 필요성 검토 후 별도 분할·새 학습 namespace로 진행합니다.